# Missing HESE events: v2 vs HESE12

1. Load v2 EvtGen events.
2. Apply 60 TeV reco_energy cut.
3. Load `HESE12.hdf5`.
4. Compare the two datasets event-by-event.

In [52]:
import tables
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load v2 EvtGen events

In [53]:
BASE_PATH = "/data/user/tvaneede/GlobalFit/reco_processing/data/hese/output/{version}"

DATASETS = [
    "IC79_2010",
    "IC86_2011",
    "IC86_2012",
    "IC86_2013",
    "IC86_2014",
    "IC86_2015",
    "IC86_2016",
    "IC86_2017",
    "IC86_2018",
    "IC86_2019",
    "IC86_2020",
    "IC86_2021",
    # "IC86_2022",
]


def load_evtgen(version):
    records = []
    for dataset in DATASETS:
        path = Path(BASE_PATH.format(version=version)) / dataset / "EvtGen" / "EvtGen.h5"
        if not path.exists():
            print(f"  {dataset}: MISSING")
            continue
        with tables.open_file(str(path), "r") as f:
            df_hdr = pd.DataFrame({
                "run":   f.root.I3EventHeader.col("Run"),
                "event": f.root.I3EventHeader.col("Event"),
                "mjd":   f.root.I3EventHeader.col("time_start_mjd"),
            })
            df_en = pd.DataFrame({
                "run":         f.root.RecoETot.col("Run"),
                "event":       f.root.RecoETot.col("Event"),
                "reco_energy": f.root.RecoETot.col("value"),
            })
            if hasattr(f.root, "HESE_CausalQTot"):
                df_qtot = pd.DataFrame({
                    "run":              f.root.HESE_CausalQTot.col("Run"),
                    "event":            f.root.HESE_CausalQTot.col("Event"),
                    "hese_causal_qtot": f.root.HESE_CausalQTot.col("value"),
                })
            else:
                df_qtot = None
            # prefer HESE_VHESelfVeto, fall back to VHESelfVeto
            veto_node = (
                f.root.HESE_VHESelfVeto if hasattr(f.root, "HESE_VHESelfVeto")
                else f.root.VHESelfVeto if hasattr(f.root, "VHESelfVeto")
                else None
            )
            if veto_node is not None:
                df_veto = pd.DataFrame({
                    "run":          veto_node.col("Run"),
                    "event":        veto_node.col("Event"),
                    "vheselfveto":  veto_node.col("value"),
                })
            else:
                df_veto = None
        df = df_hdr.merge(df_en, on=["run", "event"], how="left")
        if df_qtot is not None:
            df = df.merge(df_qtot, on=["run", "event"], how="left")
        if df_veto is not None:
            df = df.merge(df_veto, on=["run", "event"], how="left")
        df.insert(0, "dataset", dataset)
        records.append(df)
        print(f"  {dataset}: {len(df)} events")
    result = pd.concat(records, ignore_index=True)
    print(f"  Total: {len(result)}")
    return result


print("=== v2 ===")
df_v2 = load_evtgen("v2")

=== v2 ===
  IC79_2010: 7 events
  IC86_2011: 21 events
  IC86_2012: 9 events
  IC86_2013: 16 events
  IC86_2014: 15 events
  IC86_2015: 9 events
  IC86_2016: 22 events
  IC86_2017: 18 events
  IC86_2018: 16 events
  IC86_2019: 19 events
  IC86_2020: 10 events
  IC86_2021: 12 events
  Total: 174


## 2. Apply 60 TeV reco_energy cut

In [54]:
ENERGY_CUT = 60e3  # GeV

df_v2_cut = df_v2[df_v2["reco_energy"] > ENERGY_CUT].reset_index(drop=True)
print(f"v2 before cut: {len(df_v2)}")
print(f"v2 after cut:  {len(df_v2_cut)}")
df_v2_cut.groupby("dataset").size().rename("n_events")

v2 before cut: 174
v2 after cut:  103


dataset
IC79_2010     3
IC86_2011    12
IC86_2012     4
IC86_2013    12
IC86_2014    11
IC86_2015     7
IC86_2016    13
IC86_2017    12
IC86_2018     7
IC86_2019     8
IC86_2020     7
IC86_2021     7
Name: n_events, dtype: int64

## 3. Load HESE12.hdf5

In [55]:
HESE12_PATH = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr/HESE12.hdf5"

with tables.open_file(HESE12_PATH, "r") as f:
    df_hdr = pd.DataFrame({
        "run":   f.root.I3EventHeader.col("Run"),
        "event": f.root.I3EventHeader.col("Event"),
    })
    df_en = pd.DataFrame({
        "run":         f.root.RecoETot.col("Run"),
        "event":       f.root.RecoETot.col("Event"),
        "reco_energy": f.root.RecoETot.col("value"),
    })

df_hese12 = df_hdr.merge(df_en, on=["run", "event"], how="left")
print(f"HESE12.hdf5: {len(df_hese12)} events")

HESE12.hdf5: 97 events


## 4. Event-level comparison

In [56]:
set_v2_cut = set(zip(df_v2_cut["run"], df_v2_cut["event"]))
set_hese12 = set(zip(df_hese12["run"], df_hese12["event"]))

print(f"v2 (after cut): {len(set_v2_cut)}")
print(f"HESE12.hdf5:    {len(set_hese12)}")
print(f"In both:        {len(set_v2_cut & set_hese12)}")
print(f"Only in v2:     {len(set_v2_cut - set_hese12)}")
print(f"Only in HESE12: {len(set_hese12 - set_v2_cut)}")

v2 (after cut): 103
HESE12.hdf5:    97
In both:        95
Only in v2:     8
Only in HESE12: 2


In [57]:
# Per-dataset summary
df_v2_cut["in_hese12"] = [
    k in set_hese12 for k in zip(df_v2_cut["run"], df_v2_cut["event"])
]
summary = df_v2_cut.groupby("dataset")["in_hese12"].agg(n_events="count", n_in_hese12="sum")
summary["fraction"] = summary["n_in_hese12"] / summary["n_events"]
print(summary.to_string())

           n_events  n_in_hese12  fraction
dataset                                   
IC79_2010         3            3  1.000000
IC86_2011        12           11  0.916667
IC86_2012         4            3  0.750000
IC86_2013        12           10  0.833333
IC86_2014        11           10  0.909091
IC86_2015         7            6  0.857143
IC86_2016        13           12  0.923077
IC86_2017        12           12  1.000000
IC86_2018         7            7  1.000000
IC86_2019         8            8  1.000000
IC86_2020         7            6  0.857143
IC86_2021         7            7  1.000000


In [58]:
# v2 events (after cut) not in HESE12
cols = ["dataset", "run", "event", "mjd", "reco_energy"]
if "hese_causal_qtot" in df_v2_cut.columns:
    cols.append("hese_causal_qtot")
if "vheselfveto" in df_v2_cut.columns:
    cols.append("vheselfveto")

only_v2 = df_v2_cut[~df_v2_cut["in_hese12"]][cols].reset_index(drop=True)
print(f"v2 events (after cut) not in HESE12: {len(only_v2)}")
display(only_v2)

v2 events (after cut) not in HESE12: 8


,dataset,run,event,mjd,reco_energy,hese_causal_qtot,vheselfveto
0,IC86_2011,119311,430943,55928.284391,3.269002e+05,14156.095864,0
1,IC86_2012,121947,7181486,56352.807020,7.449678e+04,9355.539456,0
2,IC86_2013,123770,442256,56666.811605,4.169044e+06,6157.379055,0
3,IC86_2013,122604,17469985,56470.110391,2.250298e+05,18581.193399,0
4,IC86_2014,125826,470241,57027.490886,4.164990e+05,38785.083770,0
5,IC86_2015,127751,927145,57479.643371,2.326454e+05,8745.913758,0
6,IC86_2016,128672,38561326,57695.380221,8.313672e+04,7517.631427,0
7,IC86_2020,134994,1103075,59258.778062,3.565653e+05,10871.700004,0


# HESE 6

One of these events 
- 3	IC86_2013	122604	17469985	56470.110391

are in HESE6:
```
/data/ana/Diffuse/HESE_tau_6years/hd5/events_SPE_corrected.txt
/data/ana/Diffuse/HESE_tau_6years/hd5/eventsummary.txt
```

In [59]:
# HESE12 events not in v2 (after cut)
df_hese12["in_v2_cut"] = [
    k in set_v2_cut for k in zip(df_hese12["run"], df_hese12["event"])
]
only_hese12 = df_hese12[~df_hese12["in_v2_cut"]][["run", "event", "reco_energy"]].reset_index(drop=True)
print(f"HESE12 events not in v2 (after cut): {len(only_hese12)}")
display(only_hese12)

HESE12 events not in v2 (after cut): 2


,run,event,reco_energy
0,125914,75630389,70976.904502
1,127210,51027948,60806.979712


One event dropped out because of a lower CausalQTot in my processing:
```
0	125914	75630389	70976.904502
```

the other because of the energy, as seen in the next section

## 5. HESE12-only events — are they in v2 before the cut?

In [60]:
set_v2_all = set(zip(df_v2["run"], df_v2["event"]))

only_hese12["in_v2_precut"] = [
    (r, e) in set_v2_all for r, e in zip(only_hese12["run"], only_hese12["event"])
]

# Merge in v2 reco_energy, mjd, and vheselfveto for events that are present pre-cut
lookup_cols = ["run", "event", "mjd", "reco_energy"]
if "vheselfveto" in df_v2.columns:
    lookup_cols.append("vheselfveto")
df_v2_lookup = df_v2[lookup_cols].rename(columns={"reco_energy": "v2_reco_energy"})
only_hese12 = only_hese12.merge(df_v2_lookup, on=["run", "event"], how="left")

display_cols = ["run", "event", "reco_energy", "in_v2_precut", "mjd", "v2_reco_energy"]
if "vheselfveto" in only_hese12.columns:
    display_cols.append("vheselfveto")

print("HESE12 events not in v2 (after cut) — are they in v2 before the cut?")
display(only_hese12[display_cols])

HESE12 events not in v2 (after cut) — are they in v2 before the cut?


,run,event,reco_energy,in_v2_precut,mjd,v2_reco_energy,vheselfveto
0,125914,75630389,70976.904502,False,NaN,NaN,NaN
1,127210,51027948,60806.979712,True,57363.442334,57664.093493,0.0


## 6. v2-only events (after cut) — file presence check

In [61]:
FOLDER_ALL   = Path("/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/HESE")
FOLDER_HESE12 = Path("/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/i3files/NoDeepCore/HESE12/Bfr")
FOLDER_HESE7  = Path("/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/i3files/NoDeepCore/HESE7/Bfr")


def has_run_file(base_folder, dataset, run):
    folder = base_folder / dataset
    if not folder.exists():
        return False
    return any(folder.glob(f"Run{run:08d}*"))


for folder_name, folder in [("all", FOLDER_ALL), ("HESE12", FOLDER_HESE12), ("HESE7", FOLDER_HESE7)]:
    only_v2[f"in_{folder_name}"] = [
        has_run_file(folder, ds, r) for ds, r in zip(only_v2["dataset"], only_v2["run"])
    ]

print("v2 events (after cut) not in HESE12 — file presence in source folders:")
display(only_v2)

v2 events (after cut) not in HESE12 — file presence in source folders:


,dataset,run,event,mjd,reco_energy,hese_causal_qtot,vheselfveto,in_all,in_HESE12,in_HESE7
0,IC86_2011,119311,430943,55928.284391,3.269002e+05,14156.095864,0,False,False,False
1,IC86_2012,121947,7181486,56352.807020,7.449678e+04,9355.539456,0,True,False,False
2,IC86_2013,123770,442256,56666.811605,4.169044e+06,6157.379055,0,False,False,False
3,IC86_2013,122604,17469985,56470.110391,2.250298e+05,18581.193399,0,False,True,True
4,IC86_2014,125826,470241,57027.490886,4.164990e+05,38785.083770,0,True,False,False
5,IC86_2015,127751,927145,57479.643371,2.326454e+05,8745.913758,0,False,False,False
6,IC86_2016,128672,38561326,57695.380221,8.313672e+04,7517.631427,0,True,True,True
7,IC86_2020,134994,1103075,59258.778062,3.565653e+05,10871.700004,0,False,False,False


## 7. v2-only events (after cut) — good run list check

In [65]:
GRL_BASE = "/data/exp/IceCube/{year}/filtered/{level}/{dataset}_GoodRunInfo.txt"

_grl_cache = {}

def load_grl(dataset):
    if dataset in _grl_cache:
        return _grl_cache[dataset]
    year = int(dataset.split("_")[1])
    level = "level2" if year >= 2017 else "level2pass2a"
    path = GRL_BASE.format(year=year, level=level, dataset=dataset)
    # Columns: 0=RunNum 1=Good_i3 2=Good_it 3=LiveTime(s) 4=ActiveStrings
    #          5=ActiveDoms 6=ActiveInIce 7=OutDir 8=Comment(s)
    # usecols avoids the parser error from spaces in the Comment(s) column.
    # skiprows=[1] drops the sub-header line "(1=good 0=bad)".
    df = pd.read_csv(
        path, sep=r"\s+", skiprows=[1],
        usecols=[0, 1, 3, 4, 5, 6],
        names=["RunNum", "Good_i3", "livetime_s", "active_strings", "active_doms", "active_inIce"],
        header=0,
    )
    _grl_cache[dataset] = df
    return df


def grl_status(dataset, run):
    df = load_grl(dataset)
    row = df[df["RunNum"] == run]
    if row.empty:
        return False, None, None, None, None, None
    r = row.iloc[0]
    return True, int(r["Good_i3"]), float(r["livetime_s"]), int(r["active_strings"]), int(r["active_doms"]), int(r["active_inIce"])


results = [grl_status(ds, r) for ds, r in zip(only_v2["dataset"], only_v2["run"])]
only_v2["in_grl"], only_v2["good_i3"], only_v2["livetime_s"], only_v2["active_strings"], only_v2["active_doms"], only_v2["active_inIce"] = zip(*results)

print("v2 events (after cut) not in HESE12 — good run list status (Good_i3: 1=good, 0=bad):")
display(only_v2[["dataset", "run", "event", "mjd", "reco_energy", "hese_causal_qtot", "vheselfveto",
                  "in_grl", "good_i3", "livetime_s", "active_strings", "active_doms", "active_inIce"]])

v2 events (after cut) not in HESE12 — good run list status (Good_i3: 1=good, 0=bad):


,dataset,run,event,mjd,reco_energy,hese_causal_qtot,vheselfveto,in_grl,good_i3,livetime_s,active_strings,active_doms,active_inIce
0,IC86_2011,119311,430943,55928.284391,3.269002e+05,14156.095864,0,True,1,618.71,64,3994,3746
1,IC86_2012,121947,7181486,56352.807020,7.449678e+04,9355.539456,0,True,1,28803.88,86,5387,5044
2,IC86_2013,123770,442256,56666.811605,4.169044e+06,6157.379055,0,True,1,831.12,47,2936,2764
3,IC86_2013,122604,17469985,56470.110391,2.250298e+05,18581.193399,0,True,1,10272.28,85,5332,4989
4,IC86_2014,125826,470241,57027.490886,4.164990e+05,38785.083770,0,True,1,3220.65,86,5020,4677
5,IC86_2015,127751,927145,57479.643371,2.326454e+05,8745.913758,0,True,1,604.69,47,2937,2765
6,IC86_2016,128672,38561326,57695.380221,8.313672e+04,7517.631427,0,True,1,28805.83,86,5386,5043
7,IC86_2020,134994,1103075,59258.778062,3.565653e+05,10871.700004,0,True,1,675.35,47,2939,2767
